# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoamenAbouhaty/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I will prioritize content items for human review using historical search-performance signals. The baseline score will give higher priority to items with stronger historical visibility and weaker engagement relative to their search position. The score is used only to rank items for decision-support, not to predict a future outcome.

### Reason code

LOW_CTR_FOR_POSITION

### Action

REVIEW_CONTENT

Verdict: CONFIRMED

The observed buckets show a directional relationship between search position and CTR: CTR is higher for better average positions and lower for worse positions. This supports using CTR relative to position as a baseline decision-support signal.

In [92]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

print("DuckDB connection ready.")

DuckDB connection ready.


In [93]:
con.sql(f"""
DESCRIBE SELECT *
FROM {QUERY_90D}
""").show()

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ impressions_90d               

In [94]:
con.sql(f"""
SELECT
    CASE
        WHEN avg_position_90d <= 5 THEN '1-5'
        WHEN avg_position_90d <= 10 THEN '6-10'
        WHEN avg_position_90d <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(
        SUM(clicks_90d) * 1.0 / NULLIF(SUM(impressions_90d), 0),
        4
    ) AS ctr
FROM {QUERY_90D}
WHERE impressions_90d > 0
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN '1-5' THEN 1
        WHEN '6-10' THEN 2
        WHEN '11-20' THEN 3
        ELSE 4
    END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────┬────────┐
│ position_bucket │   n    │  ctr   │
│     varchar     │ int64  │ double │
├─────────────────┼────────┼────────┤
│ 1-5             │ 611059 │  0.004 │
│ 6-10            │ 772872 │ 0.0017 │
│ 11-20           │ 327380 │ 0.0013 │
│ 21+             │ 702937 │ 0.0004 │
└─────────────────┴────────┴────────┘



In [95]:
con.sql(f"""
SELECT
    CASE
        WHEN content_total_impressions_90d < 100 THEN '<100'
        WHEN content_total_impressions_90d < 500 THEN '100-499'
        WHEN content_total_impressions_90d < 1000 THEN '500-999'
        WHEN content_total_impressions_90d < 5000 THEN '1000-4999'
        ELSE '5000+'
    END AS volume_bucket,
    COUNT(*) AS n,
    ROUND(
        SUM(clicks_90d) * 1.0 /
        NULLIF(SUM(impressions_90d), 0),
        4
    ) AS ctr
FROM {QUERY_90D}
WHERE impressions_90d > 0
GROUP BY 1
ORDER BY
    CASE volume_bucket
        WHEN '<100' THEN 1
        WHEN '100-499' THEN 2
        WHEN '500-999' THEN 3
        WHEN '1000-4999' THEN 4
        ELSE 5
    END
""").show()

┌───────────────┬─────────┬────────┐
│ volume_bucket │    n    │  ctr   │
│    varchar    │  int64  │ double │
├───────────────┼─────────┼────────┤
│ <100          │    9215 │ 0.0005 │
│ 100-499       │   83467 │ 0.0006 │
│ 500-999       │  106003 │ 0.0009 │
│ 1000-4999     │  509618 │ 0.0016 │
│ 5000+         │ 1705945 │ 0.0023 │
└───────────────┴─────────┴────────┘



Verdict: CONFIRMED

The observed buckets show a directional relationship between content impression volume and CTR. Higher-volume content items have higher observed CTR, supporting volume as a useful baseline decision-support signal.

In [96]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("Output directory ready.")

Output directory ready.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Score design

The baseline score measures how much an item's observed CTR falls below the typical CTR for its search-position bucket. Higher scores receive higher review priority. The score uses only historical 90-day signals and is intended for decision-support ranking, not future prediction.


In [97]:
# Build the baseline ranked queue

con.sql(f"""
CREATE OR REPLACE TEMP VIEW baseline_scored AS
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        query_hash_id,
        impressions_90d,
        clicks_90d,
        avg_position_90d,
        content_total_impressions_90d,

        CASE
            WHEN avg_position_90d <= 5 THEN '1-5'
            WHEN avg_position_90d <= 10 THEN '6-10'
            WHEN avg_position_90d <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket
    FROM {QUERY_90D}
    WHERE impressions_90d > 0
),

bucket_benchmarks AS (
    SELECT
        position_bucket,
        SUM(clicks_90d) * 1.0 /
        NULLIF(SUM(impressions_90d), 0) AS expected_ctr
    FROM base
    GROUP BY position_bucket
),

scored AS (
    SELECT
        b.client_hash_id,
        b.content_hash_id,
        b.query_hash_id,
        b.impressions_90d,
        b.clicks_90d,

        ROUND(
            b.clicks_90d * 1.0 /
            NULLIF(b.impressions_90d, 0),
            6
        ) AS observed_ctr,

        b.avg_position_90d,
        b.content_total_impressions_90d,
        b.position_bucket,

        ROUND(bb.expected_ctr, 6) AS expected_ctr,

        ROUND(
            GREATEST(
                bb.expected_ctr -
                (
                    b.clicks_90d * 1.0 /
                    NULLIF(b.impressions_90d, 0)
                ),
                0
            )
            *
            LN(1 + b.impressions_90d),
            6
        ) AS score,

        'LOW_CTR_FOR_POSITION' AS reason_code,
        'REVIEW_CONTENT' AS action

    FROM base b
    JOIN bucket_benchmarks bb
        ON b.position_bucket = bb.position_bucket
)

SELECT *
FROM scored
WHERE score > 0
ORDER BY score DESC
""")

con.sql("""
COPY (
    SELECT *
    FROM baseline_scored
)
TO 'work/outputs/baseline_action_score.csv'
(FORMAT CSV, HEADER TRUE)
""")

con.sql("""
SELECT
    ROW_NUMBER() OVER (ORDER BY score DESC) AS rank,
    content_hash_id,
    query_hash_id,
    impressions_90d,
    clicks_90d,
    observed_ctr,
    avg_position_90d,
    position_bucket,
    expected_ctr,
    score,
    reason_code,
    action
FROM baseline_scored
ORDER BY score DESC
LIMIT 20
""").show()

print("Ranked queue written to work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬──────────────────────────┬────────────────────────┬─────────────────┬────────────┬──────────────┬──────────────────────┬─────────────────┬──────────────┬──────────┬──────────────────────┬────────────────┐
│ rank  │     content_hash_id      │     query_hash_id      │ impressions_90d │ clicks_90d │ observed_ctr │   avg_position_90d   │ position_bucket │ expected_ctr │  score   │     reason_code      │     action     │
│ int64 │         varchar          │        varchar         │      int64      │   int64    │    double    │        double        │     varchar     │    double    │  double  │       varchar        │    varchar     │
├───────┼──────────────────────────┼────────────────────────┼─────────────────┼────────────┼──────────────┼──────────────────────┼─────────────────┼──────────────┼──────────┼──────────────────────┼────────────────┤
│     1 │ content_943dc881428182b8 │ query_1e12d78d0219e482 │          543044 │         55 │     0.000101 │   1.6458371697321028 │ 1-5      

In [98]:
import os

print(os.path.exists("work/outputs/baseline_action_score.csv"))
print(os.path.getsize("work/outputs/baseline_action_score.csv"), "bytes")

True
355957379 bytes


In [99]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Top-20 review

The top 20 items are ranked by the baseline score, which combines the observed CTR gap against the position-bucket benchmark with an evidence-weighting term based on historical impressions.

For each item, the baseline recommends `REVIEW_CONTENT` with reason code `LOW_CTR_FOR_POSITION`.

The highest-ranked items have substantial historical impression volume, providing stronger evidence for review. However, the ranking remains a decision-support baseline rather than a definitive diagnosis.

A pick could be wrong if the low observed CTR is caused by tracking issues, query-specific behavior, unusual search intent, or other content and query context not represented by the baseline. Zero observed clicks should therefore be treated as a review signal, not proof of a content problem.

The confidence assessment is based on the amount of historical impression evidence available for each item.



### Top-20 review summary

The baseline ranks these 20 items highest because their observed CTR is below the expected CTR for the 1-5 position bucket.

All 20 items receive the action `REVIEW_CONTENT` and the reason code `LOW_CTR_FOR_POSITION`.

The review confidence is `HIGHER` for all 20 items because they have substantial historical impression evidence.

The main failure modes are:
- zero observed clicks may reflect tracking issues or query-specific effects;
- low CTR may be explained by additional content or query context not represented in the baseline;
- the baseline is a prioritization rule, not a definitive diagnosis.

Therefore, the Top-20 list should be treated as a human-review queue rather than proof that these content items require changes.

In [100]:
# Build and print the Top-20 review

top20_review = con.sql("""
SELECT
    ROW_NUMBER() OVER (ORDER BY score DESC) AS rank,
    content_hash_id,
    query_hash_id,
    impressions_90d,
    clicks_90d,
    observed_ctr,
    avg_position_90d,
    position_bucket,
    expected_ctr,
    score,
    reason_code,
    action,

    CASE
        WHEN impressions_90d < 20 THEN 'LOW'
        WHEN impressions_90d < 50 THEN 'MEDIUM'
        ELSE 'HIGHER'
    END AS confidence_note,

    CASE
        WHEN impressions_90d < 50
            THEN 'Could be wrong because the item has limited historical evidence.'
        WHEN clicks_90d = 0
            THEN 'Could be wrong because zero observed clicks may reflect tracking or query-specific effects.'
        ELSE
            'Could be wrong because this baseline does not include additional content or query context.'
    END AS what_would_make_it_wrong

FROM baseline_scored
ORDER BY score DESC
LIMIT 20
""")

for row in top20_review.fetchall():
    (
        rank,
        content_hash_id,
        query_hash_id,
        impressions_90d,
        clicks_90d,
        observed_ctr,
        avg_position_90d,
        position_bucket,
        expected_ctr,
        score,
        reason_code,
        action,
        confidence_note,
        what_would_make_it_wrong
    ) = row

    print(
        f"{rank}. Action: {action} | "
        f"Why: {reason_code}, observed CTR {observed_ctr:.6f} "
        f"vs position-bucket benchmark {expected_ctr:.6f}, score {score:.6f} | "
        f"Confidence: {confidence_note} | "
        f"Could be wrong: {what_would_make_it_wrong}"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Action: REVIEW_CONTENT | Why: LOW_CTR_FOR_POSITION, observed CTR 0.000101 vs position-bucket benchmark 0.003956, score 0.050908 | Confidence: HIGHER | Could be wrong: Could be wrong because this baseline does not include additional content or query context.
2. Action: REVIEW_CONTENT | Why: LOW_CTR_FOR_POSITION, observed CTR 0.000000 vs position-bucket benchmark 0.003956, score 0.049796 | Confidence: HIGHER | Could be wrong: Could be wrong because zero observed clicks may reflect tracking or query-specific effects.
3. Action: REVIEW_CONTENT | Why: LOW_CTR_FOR_POSITION, observed CTR 0.000000 vs position-bucket benchmark 0.003956, score 0.048469 | Confidence: HIGHER | Could be wrong: Could be wrong because zero observed clicks may reflect tracking or query-specific effects.
4. Action: REVIEW_CONTENT | Why: LOW_CTR_FOR_POSITION, observed CTR 0.000000 vs position-bucket benchmark 0.003956, score 0.047292 | Confidence: HIGHER | Could be wrong: Could be wrong because zero observed clicks m

In [101]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Leakage check

The baseline uses historical 90-day search-performance signals only. The dataset contains 21 columns, and no columns were identified with names suggesting future-window or label-derived inputs.

Result: **PASS**

No future-window or label-derived feature was used in the baseline scoring rule.


In [102]:
# Inspect weak picks

con.sql("""
SELECT
    ROW_NUMBER() OVER (ORDER BY score ASC) AS rank_from_bottom,
    content_hash_id,
    query_hash_id,
    impressions_90d,
    clicks_90d,
    observed_ctr,
    avg_position_90d,
    position_bucket,
    expected_ctr,
    score,
    reason_code,
    action
FROM baseline_scored
ORDER BY score ASC
LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────────────┬────────────────────────┬─────────────────┬────────────┬──────────────┬────────────────────┬─────────────────┬──────────────┬─────────┬──────────────────────┬────────────────┐
│ rank_from_bottom │     content_hash_id      │     query_hash_id      │ impressions_90d │ clicks_90d │ observed_ctr │  avg_position_90d  │ position_bucket │ expected_ctr │  score  │     reason_code      │     action     │
│      int64       │         varchar          │        varchar         │      int64      │   int64    │    double    │       double       │     varchar     │    double    │ double  │       varchar        │    varchar     │
├──────────────────┼──────────────────────────┼────────────────────────┼─────────────────┼────────────┼──────────────┼────────────────────┼─────────────────┼──────────────┼─────────┼──────────────────────┼────────────────┤
│                1 │ content_ac235601032be523 │ query_4e9c46a62fa8b573 │            1179 │          2 │     

In [103]:
# Check whether zero-score items enter the actionable queue

con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN score = 0 THEN 1 ELSE 0 END) AS zero_score_rows,
    SUM(CASE WHEN score > 0 THEN 1 ELSE 0 END) AS positive_score_rows
FROM baseline_scored
""").show()

┌────────────┬─────────────────┬─────────────────────┐
│ total_rows │ zero_score_rows │ positive_score_rows │
│   int64    │     int128      │       int128        │
├────────────┼─────────────────┼─────────────────────┤
│    2231641 │               0 │             2231641 │
└────────────┴─────────────────┴─────────────────────┘



In [104]:
# Check actionable baseline rows only

con.sql("""
SELECT
    COUNT(*) AS actionable_rows,
    MIN(score) AS minimum_positive_score,
    MAX(score) AS maximum_score
FROM baseline_scored
WHERE score > 0
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────────┬───────────────┐
│ actionable_rows │ minimum_positive_score │ maximum_score │
│      int64      │         double         │    double     │
├─────────────────┼────────────────────────┼───────────────┤
│         2231641 │                  2e-06 │      0.050908 │
└─────────────────┴────────────────────────┴───────────────┘



In [105]:
# Leakage check

columns = con.sql(f"""
DESCRIBE SELECT *
FROM {QUERY_90D}
""").fetchall()

column_names = [row[0] for row in columns]

future_or_label_terms = [
    "future",
    "label",
    "target",
    "outcome",
    "conversion",
    "next",
    "7d",
    "14d",
    "30d_future"
]

possible_leakage = [
    col for col in column_names
    if any(term in col.lower() for term in future_or_label_terms)
]

print("Columns checked:", len(column_names))
print("Possible future/label-derived columns:", possible_leakage)

Columns checked: 21
Possible future/label-derived columns: []


In [106]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [107]:
# Final self-check

final_check = con.sql("""
SELECT
    COUNT(*) AS ranked_rows,
    SUM(CASE WHEN score <= 0 THEN 1 ELSE 0 END) AS non_positive_scores,
    COUNT(DISTINCT action) AS action_count,
    COUNT(DISTINCT reason_code) AS reason_code_count
FROM baseline_scored
""").fetchone()

print("Ranked rows:", final_check[0])
print("Non-positive scores:", final_check[1])
print("Distinct actions:", final_check[2])
print("Distinct reason codes:", final_check[3])

print("CSV output: work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked rows: 2231641
Non-positive scores: 0
Distinct actions: 1
Distinct reason codes: 1
CSV output: work/outputs/baseline_action_score.csv


### Self-check

* The notebook builds the baseline queue from historical 90-day signals only.
* The two audited signals have documented bucket tables and verdicts.
* The ranked queue contains 2,231,641 actionable rows with positive scores.
* No non-positive scores remain in the final ranked queue.
* The queue uses one action: `REVIEW_CONTENT`.
* The queue uses one reason code: `LOW_CTR_FOR_POSITION`.
* The leakage check found no columns suggesting future-window or label-derived inputs.
* The ranked CSV is regenerated from the notebook at `work/outputs/baseline_action_score.csv`.

The baseline is intended for decision support and prioritization, not as a predictive model or definitive diagnosis.
